In [1]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

with open("../top_100_universe.txt", "r") as f:
    ACTIVE_SET = [s for s in [line.strip() for line in f] if s not in ['CRCL', 'OMDA']]
START = "2023-06-01"
END = "2025-06-01"
DATASET = "XNAS.ITCH"

df_bars = (
    adapter.scan_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
    .select(["ts_event", "symbol", "close"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
    .filter(
        pl.col("symbol").is_in(ACTIVE_SET)
    )
    .sort(["ts_event", "symbol"])
)

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] scan_parquet: 548cfb7cc98703d7d8a8a14fe79dfe1a.parquet


In [2]:
CAX = (
     pl.read_csv("../CAX/corporate_actions.csv")
     .sort(["date", "symbol"])
     .with_columns(
         pl.col("date").str.to_datetime(time_unit="ns", time_zone="UTC"),
         pl.col("symbol").cast(pl.Categorical)
     )
)
CAX

symbol,date,dividend,split
cat,"datetime[ns, UTC]",f64,f64
"""AGG""",2023-06-01 00:00:00 UTC,0.258,0.0
"""AJG""",2023-06-01 00:00:00 UTC,0.55,0.0
"""ALL""",2023-06-01 00:00:00 UTC,0.89,0.0
"""ANGL""",2023-06-01 00:00:00 UTC,0.131,0.0
"""BAC""",2023-06-01 00:00:00 UTC,0.22,0.0
…,…,…,…
"""FIS""",2026-06-11 00:00:00 UTC,0.44,0.0
"""FNV""",2026-06-11 00:00:00 UTC,0.44,0.0
"""JXN""",2026-06-11 00:00:00 UTC,0.9,0.0


In [3]:
daily_px = (
      df_bars
      .collect(engine="streaming")
    #   .filter(pl.col("symbol").is_in(ACTIVE_SET))
      .join(CAX, left_on=["ts_event", "symbol"], right_on=["date", "symbol"], how="left")
      .sort(["symbol", "ts_event"])
      .with_columns(
          pl.col("dividend").fill_null(0.0),
          # 0.0 / null == "no split" -> ratio of 1.0, not 0.0
          pl.when(pl.col("split").fill_null(0.0) == 0.0)
            .then(1.0).otherwise(pl.col("split")).alias("split"),
      )
      .with_columns(
          (pl.col("split") / (1 - pl.col("dividend") / pl.col("close").shift().over("symbol"))).alias("CAX")
      )
      .with_columns(pl.col("CAX").fill_null(1.0))            # first row per symbol: no prev_close
      .with_columns(pl.col("CAX").cum_prod().over("symbol"))
  )
daily_px

ts_event,symbol,close,dividend,split,CAX
"datetime[ns, UTC]",cat,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,"""AAPL""",180.58,0.0,1.0,1.0
2023-06-02 00:00:00 UTC,"""AAPL""",181.1,0.0,1.0,1.0
2023-06-05 00:00:00 UTC,"""AAPL""",179.12,0.0,1.0,1.0
2023-06-06 00:00:00 UTC,"""AAPL""",179.04,0.0,1.0,1.0
2023-06-07 00:00:00 UTC,"""AAPL""",177.9,0.0,1.0,1.0
…,…,…,…,…,…
2025-05-23 00:00:00 UTC,"""XLV""",130.01,0.0,1.0,1.032805
2025-05-27 00:00:00 UTC,"""XLV""",132.17,0.0,1.0,1.032805
2025-05-28 00:00:00 UTC,"""XLV""",131.18,0.0,1.0,1.032805


In [4]:
adj_px = (
    daily_px
    .with_columns((pl.col("close") * pl.col("CAX")).alias("adj_close"))
    .select(["ts_event", "symbol", "adj_close"])
)
adj_px

ts_event,symbol,adj_close
"datetime[ns, UTC]",cat,f64
2023-06-01 00:00:00 UTC,"""AAPL""",180.58
2023-06-02 00:00:00 UTC,"""AAPL""",181.1
2023-06-05 00:00:00 UTC,"""AAPL""",179.12
2023-06-06 00:00:00 UTC,"""AAPL""",179.04
2023-06-07 00:00:00 UTC,"""AAPL""",177.9
…,…,…
2025-05-23 00:00:00 UTC,"""XLV""",134.275
2025-05-27 00:00:00 UTC,"""XLV""",136.50586
2025-05-28 00:00:00 UTC,"""XLV""",135.483383


In [5]:
daily_ret = (
    adj_px
    .with_columns(
        pl.col("adj_close").pct_change().over("symbol").alias("ret")
    )
    .pivot(
        on="symbol",
        on_columns=ACTIVE_SET,
        index="ts_event",
        values="ret"
    )
    .with_columns(
        pl.all().exclude("ts_event").fill_null(pl.all().exclude("ts_event").mean())
    )
    .with_columns(
        (pl.all().exclude("ts_event") - pl.all().exclude("ts_event").mean()).round(6)
    )
)
daily_ret

ts_event,NVDA,TSLA,SPY,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,TQQQ,TLT,AMD,IWM,COST,COIN,APP,MU,IBIT,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,SQQQ,BKNG,MRVL,AMAT,TMUS,…,XLI,LQD,XLF,TSLL,SOXX,IVV,VRTX,MELI,NXPI,PYPL,SHOP,ABNB,REGN,XLE,MCHP,XLK,MDLZ,JPM,UBER,RSP,XLV,ADP,SOXL,LULU,SNPS,CDNS,WMT,MPWR,CME,UAL,FTNT,ORLY,ARM,NVDL,TTD,BAC,EA
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0
2023-06-02 00:00:00 UTC,-0.021408,0.026902,0.013491,0.006259,0.00249,0.004946,0.006565,-0.004621,0.007726,0.043334,-0.008817,-0.000188,0.007765,-0.011219,0.018855,-0.012055,-0.020249,0.036361,0.00627,0.014123,0.004703,0.001497,-0.0,-0.042313,-0.003653,-0.003304,-0.044435,0.003118,0.008344,0.023769,0.023035,-0.029928,0.024091,-0.0215,0.000356,-0.063681,…,0.026833,-0.005385,0.020129,0.03823,-0.001564,0.012713,0.027102,0.006266,0.009687,0.010431,-0.000287,0.035546,0.014157,0.027814,0.013071,0.003472,0.012514,0.0162,0.033294,0.02268,0.012584,0.025227,-0.011704,-0.024806,-0.012514,-0.011006,0.006611,-0.01993,0.029494,0.022532,-0.007559,-0.004455,0.0,-0.033473,-0.020686,0.027936,0.002752
2023-06-05 00:00:00 UTC,-0.009008,0.008421,-0.003998,-0.002275,-0.011323,0.000288,0.006263,-0.008301,0.00974,-0.014004,0.054365,-0.086325,0.010169,0.003448,-0.006002,0.00095,0.001576,-0.015992,0.008982,-0.098401,0.034828,-0.023388,-0.0,-0.004415,-0.027608,-0.032469,0.012681,0.002703,-0.053782,-0.033898,-0.006,-0.002291,0.019152,-0.040247,-0.014688,0.008408,…,-0.003835,0.00019,-0.006529,0.013547,-0.021993,-0.002919,-0.005456,0.010916,-0.036455,0.00906,0.045673,-0.012395,0.014138,-0.006023,-0.03484,-0.009121,-0.000955,-0.010848,0.015976,-0.010084,0.003215,-0.009049,-0.059227,-0.019544,-0.003489,0.008017,0.006478,-0.008489,0.001661,-0.008723,0.045918,0.005813,0.0,-0.01247,0.016468,-0.004999,-0.007961
2023-06-06 00:00:00 UTC,-0.008403,0.03989,0.003539,0.001448,-0.000836,-0.007403,0.010126,0.00153,0.009549,-0.021866,0.00906,0.072786,0.009495,-0.009208,0.004334,0.007228,0.059541,0.028644,-0.003343,-0.09896,-0.10432,0.000812,-0.0,0.058019,0.031883,0.001779,-0.00234,-0.00807,0.041037,0.002806,-0.002603,-0.013789,0.011442,0.029326,0.00315,-0.023548,…,0.000826,0.001123,0.011221,0.057795,0.016399,0.00251,-0.003788,-0.025783,0.030678,0.00835,0.059295,0.023964,-0.001843,0.008307,0.036095,0.002282,-0.017378,0.001228,-0.006362,0.012971,-0.008728,0.004761,0.05193,0.002094,-0.008594,-0.014472,-0.002531,0.021808,-0.008118,0.010934,-0.024115,-0.019518,0.0,-0.018094,-0.004085,0.022317,0.004262
2023-06-07 00:00:00 UTC,-0.029175,-0.010576,-0.005115,-0.018914,-0.006757,-0.032833,-0.041243,-0.034214,-0.037731,0.003366,-0.065435,-0.040639,-0.039038,-0.002475,-0.056331,-0.015617,-0.054513,0.018387,-0.012039,0.001993,-0.083974,-0.006868,-0.0,0.011281,-0.010701,0.012303,-0.031776,-0.001098,0.009376,0.015905,-0.034265,0.047089,-0.028892,-0.002328,0.009274,-0.014644,…,0.014911,-0.008758,0.002593,-0.016139,-0.003339,-0.004535,-0.017537,-0.035526,0.026533,-0.017699,-0.076487,-0.017193,-0.013211,0.025973,0.04516,-0.017172,-0.010424,0.007968,-0.032979,0.004762,-0.004173,-0.002008,-0.007281,-0.007836,-0.013891,-0.034114,0.001411,-0.007091,-0.013086,0.010986,-0.042284,-0.00556,0.0,-0.053997,-0.018941,0.0074,-0.01057
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-05-23 00:00:00 UTC,-0.015441,-0.0

In [6]:
# causal_vol = daily_ret.with_columns(pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1))
# st_ret = daily_ret.select(pl.all().exclude("ts_event")) / causal_vol.select(pl.all().exclude("ts_event"))
# st_ret


# floor the vol so a collapsed forecast can't produce a divide-by-~0
# (cross-sectional: e.g. 1st percentile of that day's vols, or a small abs floor)
causal_vol = daily_ret.with_columns(
    pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1)
)
VOL_FLOOR = 1e-3   # ~0.1%/day
st_ret = (
    daily_ret.select(pl.all().exclude("ts_event"))
    / causal_vol.select(pl.all().exclude("ts_event")).select(pl.all().clip(lower_bound=VOL_FLOOR))
).select(pl.all().clip(-10, 10))   # a >10σ standardized daily return is a data artifact

st_ret


NVDA,TSLA,SPY,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,TQQQ,TLT,AMD,IWM,COST,COIN,APP,MU,IBIT,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,SQQQ,BKNG,MRVL,AMAT,TMUS,PEP,…,XLI,LQD,XLF,TSLL,SOXX,IVV,VRTX,MELI,NXPI,PYPL,SHOP,ABNB,REGN,XLE,MCHP,XLK,MDLZ,JPM,UBER,RSP,XLV,ADP,SOXL,LULU,SNPS,CDNS,WMT,MPWR,CME,UAL,FTNT,ORLY,ARM,NVDL,TTD,BAC,EA
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
-10.0,10.0,10.0,6.259,2.49,4.946,6.565,-4.621,7.726,10.0,-8.817,-0.188,7.765,-10.0,10.0,-10.0,-10.0,10.0,6.27,10.0,4.703,1.497,-0.0,-10.0,-3.653,-3.304,-10.0,3.118,8.344,10.0,10.0,-10.0,10.0,-10.0,0.356,-10.0,8.266,…,10.0,-5.385,10.0,10.0,-1.564,10.0,10.0,6.266,9.687,10.0,-0.287,10.0,10.0,10.0,10.0,3.472,10.0,10.0,10.0,10.0,10.0,10.0,-10.0,-10.0,-10.0,-10.0,6.611,-10.0,10.0,10.0,-7.559,-4.455,0.0,-10.0,-10.0,10.0,2.752
-0.586075,0.435993,-0.412762,-0.506264,-6.333781,0.081103,1.328766,-2.502046,1.755922,-0.450116,8.588146,-10.0,1.824054,0.428069,-0.443374,0.109763,0.108406,-0.612587,1.995292,-9.704506,10.0,-10.0,-0.0,-0.145331,-10.0,-10.0,0.397493,1.207455,-8.977668,-1.986388,-0.362797,-0.106622,1.107287,-2.60733,-10.0,0.183901,0.832401,…,-0.199066,0.049144,-0.451778,0.49356,-10.0,-0.319806,-0.280397,2.426465,-5.241658,1.209771,10.0,-0.485687,1.39097,-0.301613,-3.712532,-3.65901,-0.106294,-0.932686,0.668348,-0.619285,0.355847,-0.499615,-7.048331,-1.097381,-0.388334,1.014573,1.364818,-0.593267,0.07844,-0.539221,8.460957,1.817412,0.0,-0.518887,1.108831,-0.249241,-4.029211
-0.624055,2.445601,0.43479,0.375443,-0.121337,-2.589461,1.905485,0.272552,1.305882,-0.829662,0.276635,1.416811,1.261014,-1.355991,0.378503,1.035759,5.079902,1.243372,-0.517938,-1.673776,-4.99057,0.058228,-0.0,2.362836,1.924781,0.091623,-0.087556,-3.343956,1.267891,0.115031,-0.189143,-0.796055,0.636535,1.087223,0.360271,-0.635026,-2.709093,…,0.052781,0.361197,0.916241,2.460917,1.249985,0.332977,-0.237181,-3.468014,1.369358,1.033364,2.181484,1.099489,-0.157165,0.505182,1.636018,0.394408,-2.399386,0.1081,-0.296818,0.900991,-1.162401,0.30677,1.447012,0.113542,-1.143889,-1.822023,-0.466692,1.736301,-0.476235,0.781074,-0.871452,-4.528724,0.0,-0.874529,-0.264513,1.361555,0.852964
-2.374099,-0.421797,-0.712416,-5.607284,-1.147544,-7.175981,-5.917776,-7.053381,-4.724116,0.133538,-2.311245,-0.700386,-4.810636,-0.328699,-5.622668,-2.215924,-1.666557,0.744459,-2.085711,0.027588,-1.462794,-0.577506,-0.0,0.305997,-0.489721,0.742308,-1.393067,-0.233817,0.268242,0.763154,-2.901946,2.865937,-1.756619,-0.084235,1.215202,-0.431731,-0.958217,…,1.117401,-3.226856,0.216538,-0.445105,-0.236927,-0.691609,-1.274973,-2.384002,1.063787,-2.170307,-1.975687,-0.767207,-1.315782,1.770188,1.694714,-3.384645,-0.948637,0.821209,-1.77612,0.33964,-0.53084,-0.149191,-0.177985,-0.497337,-1.775511,-3.359095,0.293474,-0.453275,-0.864578,0.830339,-1.580959,-0.512399,0.0,-2.698086,-1.420488,0.406475,-2.198964
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.400951,-0.185018,-0.445083,-0.537645,-0.997228,-0.414814,-0.435369,-0.544211,-0.777881,-0.401753,-0.047433,-2.020114,-0.787855,-0.267392,-0.558795,0.169457,-0.173723,-0.228726,-0.861658,-0.828469,-0.014069,-0.481423,-1.740198,-0.499992,-0.579086,-0.799101,0.667921,-0.309043,-0.69326,-0.56203,-0.770736,0.379933,-0.15557,-0.410263,-0.525021,0.27871,-0.29

In [7]:
null_columns = [
    col for col in st_ret.columns 
    if st_ret.select(pl.col(col).is_null().all()).item()
]
null_columns

[]

In [8]:
X = st_ret.drop_nulls().to_numpy()

In [9]:
# Best rank k approximation of the returns covariance matrix
U, s, Vt = np.linalg.svd(X)
k = 10
U_k, S_k, Vt_k = U[:, :k], np.diag(s[:k]), Vt[:k, :]
F = U_k
L = S_k @ Vt_k

# Residuals: what the market factor can't explain
# Adding a floor to epsilon volatility for illiquid names
eps = X - F @ L          # (T, p)
eps_var  = np.var(eps, axis=0)
EPS_FLOOR = 0.1
eps_var  = np.maximum(eps_var, EPS_FLOOR)
D = np.diag(eps_var)         # (2000, 2000) diagonal

# Factor covariance
Sigma_f = np.cov(F, rowvar=False)       # (10, 10)

# Model-implied return covariance: Σ_r = L^T Σ_f L + D
Sigma_r = L.T @ Sigma_f @ L + D         # (2000, 2000)


In [10]:
evals = np.linalg.eigvalsh(Sigma_r)

In [11]:
cond_n = np.abs(np.max(evals)) / np.abs(np.min(evals))
print(f"Condition Number: {cond_n}")

Condition Number: 430.21917323522723


In [12]:
eps_var

array([0.31843881, 0.40318469, 0.1       , 0.1       , 0.69583062,
       0.37685932, 0.4912688 , 0.5700895 , 0.37398074, 0.51203823,
       0.68355073, 0.38845526, 0.37830071, 0.79542158, 0.1       ,
       0.23165971, 0.50279872, 0.21785824, 0.59761884, 0.31294654,
       0.84187452, 0.60905692, 0.54982859, 0.74474816, 0.42554067,
       0.40780582, 0.63202095, 0.86515465, 0.76868344, 0.50365694,
       0.86482935, 1.1506685 , 0.73335889, 0.42869369, 0.29598772,
       0.63103298, 0.50259845, 0.1       , 0.78599089, 0.38504103,
       0.56137505, 0.71271974, 0.41954646, 0.66887832, 0.25418338,
       0.47712712, 1.04547848, 0.85032745, 0.1       , 0.67695676,
       0.8398705 , 0.80139495, 0.82128684, 0.36280193, 0.23814921,
       0.63049523, 0.9820968 , 0.92755197, 0.83155641, 0.67486863,
       0.71419502, 0.26933248, 0.24368645, 0.20635478, 0.2089863 ,
       0.41491745, 0.1       , 0.1       , 0.77217236, 0.81441213,
       0.33045588, 0.67750479, 0.66950337, 0.69021457, 0.78797

In [13]:
d = np.diag(Sigma_r)
print("diag  min/median/max:", d.min(), np.median(d), d.max())   # want median ≈ 1, max single digits
print("cond:", np.linalg.cond(Sigma_r))                          # want O(1e1–1e3), not 1e7

diag  min/median/max: 0.48827195278783764 1.401715951023127 1.8935574816881622
cond: 430.2191732352086


In [14]:
Sigma_r - (L.T @ Sigma_f @ L + D)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(99, 99))

In [15]:
np.save("loadings.npy", L)
np.save("factor_returns.npy", F)
np.save("idiosyncratic_diag_vec.npy", eps_var)
np.save("factor_cov_matrix.npy", Sigma_f)
np.save("cov_matrix.npy", Sigma_r)


### Modeling eps (idiosyncratic) with GARCH(1,1)

In [16]:
L[0,:]

array([ 14.05174941,  15.54040223,  24.84704926,  24.00896488,
        14.94953506,  17.68731733,  17.53348549,  15.07000026,
        15.73923649,  18.51579662,  10.57830951,  10.52501247,
        15.7428004 ,  10.38123409,  24.8961794 ,   0.9157458 ,
        14.166939  ,  19.87956757,  12.79571459,  16.19987273,
        12.34846879,  16.94357574,   6.85746389,  10.24044603,
        17.00916518,  15.73329937,  11.06360154,  11.69024692,
        15.27803076,  15.67873632,  14.56256802, -16.62199372,
        16.09474087,  14.83843511,  18.76930671,   2.55082383,
         3.75238024,  20.27376371,   9.39219734,  17.89156262,
        17.61481629,  11.04822719,  15.74439444,  15.332124  ,
        15.76739279,   1.9832086 ,   4.47640149,   9.63601301,
        24.82804268,  12.5189217 ,   7.039637  ,   1.86063566,
         8.82516645,  16.93093325,  20.51916816,  14.85434261,
        12.30553636,   5.65492412,   8.02125121,  12.44269487,
         9.57252468,  16.42899247,  19.56842982,   5.14